In [ ]:
!nvidia-smi

In [ ]:
!pip install ultralytics -q
!pip install roboflow -q

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile, os, yaml

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/dataset')

yaml_path = ''
for root, dirs, files_list in os.walk('/content/dataset'):
    for f in files_list:
        if f.endswith('.yaml'):
            yaml_path = os.path.join(root, f)

with open(yaml_path) as f:
    data = yaml.safe_load(f)
print('Classes:', data.get('names', []))

In [ ]:
import yaml, os

with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

dataset_root = os.path.dirname(yaml_path)
data['path'] = dataset_root

for key in ['train', 'val', 'test']:
    if key in data:
        p = data[key]
        if not os.path.isabs(p):
            data[key] = os.path.join(dataset_root, p)

fixed_yaml = '/content/dataset/data_fixed.yaml'
with open(fixed_yaml, 'w') as f:
    yaml.dump(data, f)

print(data)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(
    data=fixed_yaml,
    epochs=50,
    imgsz=640,
    batch=16,
    name='ppe_detector',
    patience=15,
    save=True,
    device=0
)

In [ ]:
best_model_path = '/content/runs/detect/ppe_detector/weights/best.pt'
model_val = YOLO(best_model_path)
metrics = model_val.val(data=fixed_yaml)
print(f'mAP50: {metrics.box.map50:.3f}')

In [ ]:
model_best = YOLO(best_model_path)
model_best.export(
    format='onnx',
    imgsz=640,
    simplify=True,
    opset=12
)
print('ONNX exported!')

In [ ]:
import json, yaml

with open(fixed_yaml) as f:
    d = yaml.safe_load(f)

with open('/content/classes.json', 'w') as f:
    json.dump(d['names'], f)

print('Classes:')
for i, name in enumerate(d['names']):
    print(f'  {i}: {name}')

In [ ]:
from google.colab import files

onnx_path = best_model_path.replace('.pt', '.onnx')
files.download(onnx_path)
files.download('/content/classes.json')
print('Done! Upload both files to Claude.')